# Day 8 – BeautifulSoup Web Scraping

In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [14]:
url = "https://books.toscrape.com/"
response = requests.get(url, timeout=15)

In [15]:
response.raise_for_status()

In [16]:
soup = BeautifulSoup(response.text, "html.parser")

In [17]:
print("Page title:", soup.title.get_text(strip=True))

Page title: All products | Books to Scrape - Sandbox


In [18]:
print("Number of links:", len(soup.find_all("a")))

Number of links: 94


In [19]:
records = []

for article in soup.select("article.product_pod"):
    title = article.select_one("h3 a")
    price = article.select_one(".price_color")

    if title and price:
        records.append({
            "BookTitle": title.get("title", title.get_text(strip=True)),
            "Price": price.get_text(strip=True)
        })

In [20]:
books_df = pd.DataFrame(records)

In [21]:
books_df.head()

,BookTitle,Price
0,A Light in the Attic,Â£51.77
1,Tipping the Velvet,Â£53.74
2,Soumission,Â£50.10
3,Sharp Objects,Â£47.82
4,Sapiens: A Brief History of Humankind,Â£54.23


In [22]:
books_df.shape

(20, 2)

In [23]:
books_df["Price"] = (
    books_df["Price"]
    .str.replace("Â£", "", regex=False)
    .astype(float)
)

In [24]:
books_df["BookTitle"] = books_df["BookTitle"].str.strip()

In [25]:
books_df = books_df.drop_duplicates().reset_index(drop=True)

In [26]:
books_df.head()

,BookTitle,Price
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sharp Objects,47.82
4,Sapiens: A Brief History of Humankind,54.23


In [27]:
books_df.isnull().sum()

,0
BookTitle,0
Price,0


In [28]:
links = []

for link in soup.find_all("a", href=True):
    links.append(link["href"])

links_df = pd.DataFrame({"Link": links}).drop_duplicates()

In [29]:
links_df.head(10)

,Link
0,index.html
2,catalogue/category/books_1/index.html
3,catalogue/category/books/travel_2/index.html
4,catalogue/category/books/mystery_3/index.html
5,catalogue/category/books/historical-fiction_4/...
6,catalogue/category/books/sequential-art_5/inde...
7,catalogue/category/books/classics_6/index.html
8,catalogue/category/books/philosophy_7/index.html
9,catalogue/category/books/romance_8/index.html
10,catalogue/category/books/womens-fiction_9/inde...


In [30]:
hr = pd.read_excel("/content/HR-Employee-Attrition.xlsx")

In [31]:
web_summary = pd.DataFrame({
    "DataSource": ["Books to Scrape"],
    "ScrapedRecords": [len(books_df)],
    "AveragePrice": [books_df["Price"].mean()],
    "LowestPrice": [books_df["Price"].min()],
    "HighestPrice": [books_df["Price"].max()]
})

In [32]:
len(hr)

1470

In [33]:
print(web_summary)

        DataSource  ScrapedRecords  AveragePrice  LowestPrice  HighestPrice
0  Books to Scrape              20       38.0485        13.99         57.25


In [34]:
# Save outputs
books_df.to_csv("scraped_books.csv", index=False)
links_df.to_csv("scraped_links.csv", index=False)
web_summary.to_csv("web_summary.csv", index=False)
